# NOTEBOOK 02 – NGỌC LAN | CHƯƠNG 2: CƠ SỞ LÝ THUYẾT

**Phụ trách:** giữ nguyên 2.1–2.5, đồng thời thực hành huấn luyện **Random Forest** và đo **Precision, Recall, F1, AP/PR-AUC, FP, Top-K trên Validation**. Issue **#3**. Lý thuyết 12/11, RF bàn giao trước 20/11/2026.

**Mục đích:** tạo phần lý thuyết chuẩn, có thể giải thích khi bảo vệ; phụ trách mô hình Random Forest; **không dùng Test để chọn mô hình/ngưỡng**. Trúc Chi triển khai Dummy và MLP, Vĩnh Chi đánh giá độc lập cả ba trên Test.

[Tệp Markdown lý thuyết chi tiết](../docs/theory/02_ngoc_lan_ch2.md) · [Nguồn](../docs/REFERENCES.md).

## 2.1 Cơ sở về TMĐT và logistics
Mô tả vòng đời đơn đặt hàng → phê duyệt thanh toán → chuẩn bị hàng → giao cho vận chuyển → giao đến khách.

**Khái niệm:** `estimated delivery date` = ngày giao dự kiến; `actual delivery date` = ngày giao thực tế; `SLA` = thời hạn cam kết theo điều kiện dịch vụ.

**Tại sao dự báo sớm:** bộ phận vận hành cần ưu tiên theo dõi khi còn khả năng can thiệp. Việc phân biệt **dự báo trước** và **giải thích sau** là bắt buộc.

## 2.2 Khai thác dữ liệu và EDA
1. Kiểm tra schema, độ đầy đủ, khóa chính, khóa ngoại, quan hệ 1–n.
2. Chọn đơn vị phân tích **order_id** và tạo nhãn nhị phân bằng ngày lịch.
3. Thống kê đơn, đơn trễ, phân phối giá/phí vận chuyển, hành vi theo thời gian, bang và nhóm hàng.
4. Trực quan hóa có diễn giải: biểu đồ, bảng tỷ lệ, số mẫu của từng nhóm.
5. Phân biệt tương quan trong EDA và khả năng dự báo tại `order_approved_at`.

Cần liên kết hình và kết quả EDA do Lam Linh thực nghiệm, không tự đặt số liệu.

## 2.3 Cơ sở thuật toán

**Dummy Classifier:** mốc tham chiếu để kiểm tra mô hình phức tạp có mang lại lợi ích hay không.

**Random Forest:** tập hợp nhiều cây quyết định bằng bootstrap và ngẫu nhiên hóa đặc trưng; phù hợp dữ liệu dạng bảng, có thể kiểm soát overfitting bằng min_samples_leaf/max_depth.

**MLP:** mạng nơ-ron nhiều lớp; học qua tối ưu hóa trọng số, cần mã hóa categorical và chuẩn hóa numeric, điều chỉnh architecture và regularization.

**Lưu ý học thuật:** không mặc định Deep Learning phải thắng Random Forest.

**Bổ sung thực hành 2.3:** Ngọc Lan trực tiếp `fit` Random Forest trên **Train chung** do giao thức Trúc Chi định nghĩa; xuất toàn bộ pipeline, xác suất RF trên Validation, cấu hình và metadata. Trúc Chi *không fit lại RF*; chỉ tải artifact và kiểm tra cùng `split_id`.


## 2.4 Các độ đo đánh giá

Với lớp dương = đơn trễ:
- **Precision** = TP/(TP+FP): chất lượng cảnh báo.
- **Recall** = TP/(TP+FN): tỷ lệ đơn trễ được phát hiện.
- **F1** = 2PR/(P+R): điều hòa Precision và Recall.
- **Average Precision (AP)**: trung bình có trọng số của Precision theo mức tăng Recall trên đường Precision–Recall (`average_precision_score`).
- **False Positives**: số đơn báo trễ nhưng thực tế đúng hạn.
- **Precision@K**: tỷ lệ đơn thật sự trễ trong Top-K xác suất lớn nhất.
- **Recall@K**: tỷ lệ đơn trễ được phát hiện trong Top-K, so với tổng số đơn trễ.

Không đồng nhất AP với ROC-AUC. Có thể báo cả ROC-AUC phụ, nhưng AP được ưu tiên nếu lớp dương ít.
**Bài thực hành 2.4:** từ xác suất RF trên Validation tính lại TP, FP, FN, TN, Precision, Recall, F1, AP, Precision@K và Recall@K. **Chỉ minh họa thực nghiệm với Validation**; người làm Test độc lập là Vĩnh Chi.


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

# Ví dụ NHÂN TẠO để minh họa cách tính – KHÔNG phải kết quả Olist.
y_true = np.array([0, 1, 1, 0, 0, 1])
y_prob = np.array([0.10, 0.80, 0.60, 0.30, 0.70, 0.90])
threshold = 0.50
y_pred = (y_prob >= threshold).astype(int)
print("Precision minh họa:", round(precision_score(y_true, y_pred, zero_division=0), 3))
print("Recall minh họa:", round(recall_score(y_true, y_pred, zero_division=0), 3))
print("F1 minh họa:", round(f1_score(y_true, y_pred, zero_division=0), 3))
print("Average Precision minh họa:", round(average_precision_score(y_true, y_prob), 3))
print("Confusion matrix minh họa:\n", confusion_matrix(y_true, y_pred))

### 2.4.1 Gợi ý diễn giải hình minh họa
Giải thích mối đánh đổi khi thay đổi threshold: threshold thấp thường tăng Recall nhưng có thể tăng cảnh báo sai. **Kịch bản Top-K** thể hiện khả năng giám sát có giới hạn nhân lực. Ví dụ giả định ở trên chỉ để dạy thuật toán; bắt buộc xóa hoặc ghi rõ “minh họa” khi đưa vào slide báo cáo.
Phần ví dụ giả định trước đó chỉ để học công thức. Kết quả RF Validation có được từ các cell thực nghiệm tiếp theo, lưu riêng và có thể kiểm tra lại.


## THỰC HÀNH 2.3 – RANDOM FOREST (NGỌC LAN HUẤN LUYỆN)

**Đầu vào:** `data/processed/orders_model_ready.csv` từ Lam Linh; split Train/Validation/Test thống nhất và metadata từ `src/experiment.py`.

**Thời điểm dự báo:** `order_approved_at`. **Không dùng** `order_delivered_customer_date`, status cuối hoặc reviews làm feature.

**Trách nhiệm Ngọc Lan:** huấn luyện RF và chọn cấu hình bằng **AP Validation**, sau đó chọn threshold trên **Validation**, xuất `models/ngoc_lan_random_forest.joblib`, `rf_val_predictions.csv`, `rf_metadata.json`; bàn giao cho Trúc Chi. Không huấn luyện MLP/Dummy ở notebook này.

**Kiểm tra đầu tiên:** cần có `orders_model_ready.csv`; nếu Ch3 chưa bàn giao, dừng để yêu cầu Lam Linh hoàn thành. Nếu split đã được tạo, hệ thống kiểm tra SHA và `split_id` thay vì chia lại theo cách khác.

In [ ]:
from pathlib import Path
import sys, json, joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "docs" / "DATA_CONTRACT.md").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook trong repo hoặc thư mục notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"
from src.experiment import (NUMERIC_FEATURES, CATEGORY_FEATURES, SAFE_FEATURES,
                            get_shared_splits, choose_threshold_on_validation,
                            precision_recall_at_k)
splits = get_shared_splits(PROCESSED)
train, val = splits["train"], splits["val"]
split_id = splits["metadata"]["split_id"]
print("Split dùng chung:", split_id)
display(pd.DataFrame([{"split": k, "n": len(splits[k]),
                       "late_rate": splits[k]["is_late"].mean()}
                      for k in ("train", "val", "test")]))
print("Tuyệt đối không fit hoặc đánh giá RF trên Test trong notebook Ngọc Lan.")

### 2.3.1 Xây dựng pipeline Random Forest – Fit trên Train

Dùng cùng feature whitelist với Trúc Chi. Pipeline xử lý missing, one-hot categorical trước khi `RandomForestClassifier`.

**Tham số cần trình bày:** `n_estimators`, `min_samples_leaf`, `class_weight`, `random_state`; giải thích lý do và khả năng overfitting. Các cấu hình minh họa cần được chọn từ **Validation**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

X_train = train[SAFE_FEATURES]
y_train = train["is_late"].astype(int)
X_val = val[SAFE_FEATURES]
y_val = val["is_late"].astype(int)

def rf_pipeline(min_leaf=5):
    prep = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), CATEGORY_FEATURES)
    ])
    return Pipeline([
        ("preprocess", prep),
        ("rf", RandomForestClassifier(
            n_estimators=200, min_samples_leaf=min_leaf,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42))
    ])

records, fitted = [], {}
for min_leaf in [2, 5, 10]:
    candidate = rf_pipeline(min_leaf)
    candidate.fit(X_train, y_train)  # TRAIN ONLY
    prob = candidate.predict_proba(X_val)[:, 1]  # VALIDATION ONLY
    score = float(average_precision_score(y_val, prob))
    records.append({"min_samples_leaf": min_leaf, "AP_validation": score})
    fitted[min_leaf] = candidate

cv_table = pd.DataFrame(records).sort_values(
    ["AP_validation", "min_samples_leaf"], ascending=[False, True])
display(cv_table)
best_leaf = int(cv_table.iloc[0]["min_samples_leaf"])
rf = fitted[best_leaf]
rf_val_prob = rf.predict_proba(X_val)[:, 1]
print("Đã chọn RF bằng AP trên Validation:", best_leaf, "split:", split_id)

### 2.4.1 Đánh giá Random Forest trên Validation

- Lớp dương: `is_late=1`.
- Chọn threshold bằng **F1 Validation**, nếu bằng nhau ưu tiên ít FP hơn.
- Báo **Precision, Recall, F1, AP, TP, FP, FN, TN** và so sánh Top-K.
- Kết quả ở đây chỉ là đánh giá Validation để phát triển phương pháp; **Vĩnh Chi** sẽ đánh giá chính thức trên frozen Test.

Chỉ đưa bảng giá trị thực sau khi chạy Olist. Không sao chép ví dụ nhân tạo thành kết quả nhóm.

In [ ]:
from src.experiment import classification_metrics
rf_val_metrics = choose_threshold_on_validation(y_val, rf_val_prob)
rf_threshold = float(rf_val_metrics["threshold"])
display(pd.DataFrame([{"model": "Random Forest", **rf_val_metrics,
                       "split_id": split_id, "dataset": "Validation"}]))
k_values = sorted(set(min(k, len(val)) for k in [50, 100, 200] if k > 0))
rf_topk_val = pd.DataFrame([
    precision_recall_at_k(y_val.to_numpy(), rf_val_prob, k)
    for k in k_values
])
display(rf_topk_val)
print("FP = số đơn cảnh báo sai trên Validation:",
      rf_val_metrics["FP"])

### 2.4.2 Bàn giao model RF cho Trúc Chi

**Bắt buộc:**
1. `models/ngoc_lan_random_forest.joblib`: pipeline RF đã fit bằng Train.
2. `data/processed/rf_val_predictions.csv`: `order_id`, `y_true`, `p_rf` của Validation.
3. `data/processed/rf_metadata.json`: fingerprint split, feature list, cấu hình, threshold và metrics trên Validation.
4. Có giải thích trong `reports/chapters/CH02_THEORY.md`; tách rõ thuật toán và kết quả thực nghiệm.
5. Link PR/commit và xác nhận Trúc Chi đã nhận. **Không đẩy CSV/model lớn lên GitHub** (bàn giao theo kênh nội bộ nhóm).

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
pred_val = pd.DataFrame({
    "order_id": val["order_id"].to_numpy(),
    "y_true": y_val.to_numpy(),
    "p_rf": rf_val_prob,
})
assert pred_val["order_id"].is_unique
pred_val.to_csv(PROCESSED / "rf_val_predictions.csv", index=False)
joblib.dump(rf, MODEL_DIR / "ngoc_lan_random_forest.joblib")

metadata = {
    "owner": "Ngọc Lan", "model": "rf", "split_id": split_id,
    "dataset_sha256": splits["metadata"]["csv_sha256"],
    "features": list(SAFE_FEATURES),
    "best_min_samples_leaf": best_leaf,
    "n_estimators": 200, "random_state": 42,
    "validation_metrics": rf_val_metrics,
    "validation_topk": rf_topk_val.to_dict(orient="records"),
    "validation_n": len(val),
    "threshold_origin": "Validation only"
}
(PROCESSED / "rf_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã lưu RF model, Validation predictions, metadata.")
print("Người nhận: Trúc Chi. Vĩnh Chi sẽ đánh giá Test độc lập.")

### Checklist bổ sung – Ngọc Lan
- [ ] RF chỉ fit trên Train chung; các bước encode/impute fit Train-only.
- [ ] Có bảng 3 cấu hình RF, chọn bằng AP Validation.
- [ ] Có Precision/Recall/F1/AP/FP/Top-K Validation.
- [ ] Đã bàn giao pipeline, `rf_val_predictions.csv`, `rf_metadata.json` với `split_id` và không dùng Test.
- [ ] Cập nhật GitHub Issue #3, GSheet mục 2.3–2.4 và T15.

## 2.5 Nghiên cứu liên quan
**Công việc:** lựa chọn **3–5 công trình** gần chủ đề giao hàng trễ, logistics/last-mile hoặc dự báo nhị phân trên dữ liệu bảng.

Bảng tổng quan bắt buộc có:
- Tác giả, năm và đường dẫn DOI/URL.
- Bộ dữ liệu và bối cảnh nghiên cứu.
- Mô hình, độ đo và đóng góp.
- Giới hạn và điểm khác so với đồ án của nhóm.

**Chưa có bài báo được nhóm xác minh** trong notebook này; không tự tạo tên tác giả hay kết quả từ nguồn chưa đọc.

In [ ]:
import pandas as pd
literature_template = pd.DataFrame(columns=[
    "tac_gia_nam", "tieu_de", "doi_url", "du_lieu", "mo_hinh", "do_do",
    "dong_gop", "han_che", "lien_he_de_tai"
])
display(literature_template)
print("Mẫu bảng, chưa điền tài liệu tham khảo chưa kiểm chứng.")

## 2.6 Hướng dẫn viết và nghiệm thu Chương 2
- [ ] Có định nghĩa/ký hiệu được giải thích bằng lời, không chép công thức thiếu nguồn.
- [ ] Thuật toán được mô tả **đúng với code cuối** của Trúc Chi.
- [ ] Các độ đo trùng định nghĩa và cách tính ở Chương 5 của Vĩnh Chi.
- [ ] Có 3–5 nguồn học thuật được tra cứu và xác minh.
- [ ] Mỗi mục liên kết cụ thể với đề tài Olist.
- [ ] Hoàn thành `reports/chapters/CH02_THEORY.md` và link Pull Request.

**Cấu trúc mỗi mục:** mở đầu – khái niệm – lý do sử dụng – ví dụ liên hệ Olist – hạn chế – tài liệu trích dẫn.
- [ ] Đã huấn luyện RF và có metrics Validation, artifact, metadata để Trúc Chi tích hợp.
- [ ] Không train RF trong notebook của Trúc Chi; Test được giữ kín cho Vĩnh Chi đánh giá cuối.
